In [1]:
import numpy as np
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
from joblib import load
import os
import glob
from spectral import envi
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
import json
from datetime import datetime




In [11]:
def load_reference(hdr_file, bin_file):
    """Load reference data from ENVI format files"""
    reference = envi.open(hdr_file, bin_file)
    return reference.load()

def apply_reference_correction(cube, white_ref, dark_ref):
    """
    Apply white and dark reference correction to hyperspectral cube
    Using formula: (Sample - Dark) / (White - Dark)
    """
    # Ensure references match cube dimensions
    if white_ref.shape != dark_ref.shape:
        raise ValueError("White and dark reference dimensions must match")
    
    # Broadcast references if needed
    if len(white_ref.shape) < len(cube.shape):
        white_ref = np.expand_dims(white_ref, axis=0)
        dark_ref = np.expand_dims(dark_ref, axis=0)
    
    denominator = white_ref - dark_ref    
    denominator[denominator == 0] = 1  # Avoid division by zero    
    corrected = (cube - dark_ref) / denominator    
    # Clip values to valid range [0, 1]
    corrected = np.clip(corrected, 0, 1)
    return corrected

# Load reference data, maybe change directory if needed. .hdr file comes first, then .bin or .dat
white_reference = load_reference('swiss/whiteReference.hdr', 'swiss/whiteReference.bin')
dark_reference = load_reference('swiss/darkReference.hdr', 'swiss/darkReference.bin')


def process_cube(cube):
    
    if cube.shape[2] != 301:  
        raise ValueError(f"Input cube has {cube.shape[2]} wavelengths but model expects 301 wavelengths")
         
    # First get spatial mean (across height and width)
    mean_spectrum = np.mean(cube, axis=(0, 1))
    
    # Apply preprocessing steps
    normalized = (mean_spectrum - np.min(mean_spectrum)) / (np.max(mean_spectrum) - np.min(mean_spectrum))
    smoothed = savgol_filter(normalized, 15, 3)
    denoised = gaussian_filter1d(smoothed, sigma=1)
    return denoised



def get_latest_hsi_files(directory):
    # First check for .bin files
    bin_files = glob.glob(f"{directory}/*.bin")
    
    if bin_files:
        # Get the most recent .bin file
        latest_bin = max(bin_files, key=os.path.getctime)
        base_name = os.path.splitext(latest_bin)[0]
        hdr_file = f"{base_name}.hdr"        
        print("Latest bin file:")
        print(latest_bin)
        return hdr_file, latest_bin
    
    # If no .bin files, check for .dat files
    dat_files = glob.glob(f"{directory}/*.dat")
    if dat_files:
        # Get the most recent .dat file
        latest_dat = max(dat_files, key=os.path.getctime)
        base_name = os.path.splitext(latest_dat)[0]
        hdr_file = f"{base_name}.hdr"        
        print("Latest dat file:")
        print(latest_dat)
        return hdr_file, latest_dat
    
    # If neither .bin nor .dat files are found
    raise FileNotFoundError("No .bin or .dat files found in directory")

def load_hsi_cube_of_test_sample(directory):
    try:
        header_file, data_file = get_latest_hsi_files(directory)        
        if not os.path.isfile(header_file):
            raise FileNotFoundError(f"Header file not found: {header_file}")            
        try:
            data = envi.open(header_file, data_file)
        except Exception as e:
            raise ValueError(f"Failed to open ENVI files. Possible corruption in header or data file: {str(e)}")           
        try:
            cube = data.load()
        except Exception as e:
            raise ValueError(f"Failed to load hyperspectral cube. Data may be corrupt: {str(e)}")        
        if cube is None:
            raise ValueError("Loaded cube is None")            
        if cube.size == 0:
            raise ValueError("Loaded cube is empty (zero size)")            
        if len(cube.shape) != 3:
            raise ValueError(f"Invalid cube dimensions: expected 3 dimensions, got {len(cube.shape)}")            
        
        print(f"Number of spectral bands: {cube.shape[2]} wavelengths")
        # training data shape: (55, 1020, 301) and only number of wavelength is important    
        return cube
        
    except Exception as e:
        # Catch any other unexpected errors and provide context
        raise Exception(f"Error loading hyperspectral cube from {directory}: {str(e)}") from e

cubes_test_sample = {
    'sample_1': load_hsi_cube_of_test_sample('swiss/')
}

loaded_pls = load('pls_model.joblib')
loaded_scaler = load('scaler.joblib')

new_data_corrected = apply_reference_correction(cubes_test_sample['sample_1'], white_reference, dark_reference)



new_spectrum = process_cube(new_data_corrected)
new_spectrum_scaled = loaded_scaler.transform(new_spectrum.reshape(1, -1))
new_pred = loaded_pls.predict(new_spectrum_scaled)[0]

print(f"Predicted concentration: {new_pred:.3f}\n")

output_data = {
    'prediction': float(new_pred),  
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'model': 'PLS',
        'components': 5,        
    }
}

# Save to JSON with fixed filename
with open('pls_prediction.json', 'w') as f:
    json.dump(output_data, f, indent=4)

print("Prediction saved to pls_prediction.json")


Latest bin file:
swiss/1_5_00ml.bin
Number of spectral bands: 301 wavelengths


/tmp/ipykernel_329518/3754088998.py:20: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  denominator = white_ref - dark_ref
/tmp/ipykernel_329518/3754088998.py:22: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  corrected = (cube - dark_ref) / denominator


Predicted concentration: 3.724

Prediction saved to pls_prediction.json
